# Алгоритмы

## LightGCN

### Архитектура и математическое описание по шагам

Рекомендательная система представляется в виде двудольного графа (bipartite graph) пользователей $U$ и объектов $I$. Если пользователь $u$ взаимодействовал с объектом $i$, между ними существует ребро.

#### Шаг 1: Инициализация эмбеддингов (Слой 0)

Для каждого пользователя u и объекта i создаются обучаемые векторные представления (эмбеддинги) нулевого слоя:

$e_{u}^{(0)}\in \mathbb{R}^{d},\quad e_{i}^{(0)}\in \mathbb{R}^{d}$

Где $d$ — размерность эмбеддинга. Это единственные параметры модели, которые будут обновляться методом градиентного спуска.

#### Шаг 2: Графовая свертка (Пропагационный слой)

На каждом слое $k$ эмбеддинг вершины обновляется как среднее арифметическое эмбеддингов её соседей с предыдущего слоя. Математически для пользователя u и объекта i:

$e_{u}^{(k+1)}=\sum _{i\in \mathcal{N}_{u}}\frac{1}{\sqrt{|{}\mathcal{N}_{u}|{}\cdot |{}\mathcal{N}_{i}|{}}}e_{i}^{(k)}$

$e_{i}^{(k+1)}=\sum _{u\in \mathcal{N}_{i}}\frac{1}{\sqrt{|{}\mathcal{N}_{i}|{}\cdot |{}\mathcal{N}_{u}|{}}}e_{u}^{(k)}$

- $\mathcal{N}_{u}$ — множество объектов, с которыми взаимодействовал пользователь $u$.
- $\vert{}\mathcal{N}_u\vert{}$ — степень вершины (количество связей).
- Коэффициент $\frac{1}{\sqrt{|{}\mathcal{N}_{u}|{}\cdot |{}\mathcal{N}_{i}|{}}}$ — это симметричная нормализация, которая предотвращает чрезмерное увеличение масштаба эмбеддингов у популярных объектов и активных пользователей.

В матричном виде для всех вершин сразу: $E^{(k+1)}=\tilde{A}E^{(k)}$

Где $\tilde{A} = D^{-1/2} A D^{-1/2}$ — нормализованная матрица смежности графа, а $E^{(k)}$ — матрица эмбеддингов всех пользователей и объектов на слое $k$.

#### Шаг 3: Объединение слоев (Layer Combination)

После прохождения $K$ слоев (обычно $K=3–4$), мы получаем $K+1$ матрицу представлений. Вместо того чтобы брать только последний слой, LightGCN берет взвешенную сумму эмбеддингов со всех слоев:

$e_{u}=\sum _{k=0}^{K}\alpha _{k}e_{u}^{(k)},\quad e_{i}=\sum _{k=0}^{K}\alpha _{k}e_{i}^{(k)}$

Обычно веса фиксируются как $\alpha_k = \frac{1}{K+1}$, что эквивалентно простому усреднению.

Это позволяет объединить информацию разного уровня: от исходных скрытых свойств (слой $0$) до глобальной структуры графа (слой $K$).

#### Шаг 4: Вычисление предсказания

Финальный скор (вероятность взаимодействия) между пользователем u и объектом i вычисляется как скалярное произведение их результирующих эмбеддингов: $\hat{y}_{ui}=e_{u}^{T}e_{i}$.

#### Шаг 5: Оптимизация (Функция потерь BPR)

Для обучения модели используется Bayesian Personalized Ranking (BPR) loss. Она заставляет модель максимизировать расстояние между позитивными (с которыми пользователь взаимодействовал) и негативными (случайными) объектами:

$L_{BPR}=-\sum _{u=1}^{U}\sum _{i\in \mathcal{N}_{u}}\sum _{j\notin \mathcal{N}_{u}}\ln \sigma (\hat{y}_{ui}-\hat{y}_{uj})+\lambda |{}|{}E^{(0)}|{}|{}^{2}$

Где $\sigma$ — сигмоида, а $\lambda$ — коэффициент $L_{2}$ -регуляризации для эмбеддингов нулевого слоя.